# 11 Density Route Analysis

## Purpose

This notebook connects Houston METRO route geography with Harris County population density data.

Earlier notebooks identified high-ridership routes, calculated service productivity, and created a population density map. This notebook brings those pieces together by comparing selected high-performing routes against the spatial distribution of population density.

## Inputs

- `data/processed/key_route_geometry.csv`
- `data/processed/harris_tracts_density.geojson`
- `data/processed/route_investment_metrics.csv`

## Outputs

- Density-and-route overlay visualizations
- Geographic context for final investment recommendations

## Why This Matters

Transit investment decisions are not only about ridership totals. They are also about where people live, which neighborhoods are dense enough to support frequent transit, and whether high-performing routes align with areas of strong potential demand.


## 1. Import Libraries and Load Spatial Data

This section loads the Python libraries needed for tabular and geospatial analysis, along with the processed census tract density dataset.

In [8]:
import geopandas as gpd
import pandas as pd
import os

for root, dirs, files in os.walk("../data"):
    for file in files:
        if file.endswith(".shp"):
            print(os.path.join(root, file))

## 2. Load Harris County Population Density Data

The population density file combines Census tract geometry with ACS population data. This allows the project to map where population is concentrated across Harris County.

In [3]:
tracts = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_48_tract.zip"
)

print(tracts.shape)
tracts.head()

(6896, 14)


,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,48,157,674100,48157674100,1400000US48157674100,6741,Census Tract 6741,G5020,S,4170615,0,+29.5791692,-095.5995891,"POLYGON ((-95.61467 29.57828, -95.61339 29.578..."
1,48,157,674200,48157674200,1400000US48157674200,6742,Census Tract 6742,G5020,S,5800266,189859,+29.5713421,-095.6222174,"POLYGON ((-95.63989 29.58625, -95.63974 29.586..."
2,48,441,013501,48441013501,1400000US48441013501,135.01,Census Tract 135.01,G5020,S,866463520,4539887,+32.1806265,-099.9286588,"POLYGON ((-100.15192 32.08412, -100.15188 32.0..."
3,48,441,013602,48441013602,1400000US48441013602,136.02,Census Tract 136.02,G5020,S,665184779,422010,+32.3991003,-100.0146292,"POLYGON ((-100.14955 32.2816, -100.1495 32.286..."
4,48,441,013601,48441013601,1400000US48441013601,136.01,Census Tract 136.01,G5020,S,36492558,0,+32.4639334,-100.0041804,"POLYGON ((-100.03974 32.48854, -100.03064 32.4..."


## Additional Spatial Analysis

This step supports the comparison between transit corridors and population density.

In [4]:
harris_tracts = tracts[
    tracts["COUNTYFP"] == "201"
].copy()

print(harris_tracts.shape)
harris_tracts.head()

(1115, 14)


,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
18,48,201,323701,48201323701,1400000US48201323701,3237.01,Census Tract 3237.01,G5020,S,2311208,5694,+29.6551252,-095.1769249,"POLYGON ((-95.18667 29.66507, -95.18633 29.665..."
19,48,201,241103,48201241103,1400000US48201241103,2411.03,Census Tract 2411.03,G5020,S,2782048,35540,+30.0474108,-095.3807767,"POLYGON ((-95.3915 30.05522, -95.39134 30.0553..."
20,48,201,450801,48201450801,1400000US48201450801,4508.01,Census Tract 4508.01,G5020,S,903291,17758,+29.7598221,-095.5680149,"POLYGON ((-95.57635 29.76008, -95.57634 29.760..."
21,48,201,454302,48201454302,1400000US48201454302,4543.02,Census Tract 4543.02,G5020,S,2750427,1867,+29.7153470,-095.6704755,"POLYGON ((-95.68708 29.71008, -95.68669 29.710..."
22,48,201,451902,48201451902,1400000US48201451902,4519.02,Census Tract 4519.02,G5020,S,2057671,0,+29.7225162,-095.5987045,"POLYGON ((-95.60573 29.72978, -95.60278 29.729..."


## 5. Save Outputs

Any processed datasets or figures are saved so they can be reused in the final recommendations notebook and future interactive website.

In [5]:
harris_tracts.to_file(
    "../data/processed/harris_tracts.geojson",
    driver="GeoJSON"
)

print("saved")

saved


## Additional Spatial Analysis

This step supports the comparison between transit corridors and population density.

In [9]:
acs = pd.read_csv("../data/acs_harris_tracts.csv")

## Additional Spatial Analysis

This step supports the comparison between transit corridors and population density.

In [10]:
harris_tracts["GEOID"] = harris_tracts["GEOID"].astype(str)

acs["GEOID"] = (
    acs["state"].astype(str).str.zfill(2)
    + acs["county"].astype(str).str.zfill(3)
    + acs["tract"].astype(str).str.zfill(6)
)

tracts_acs = harris_tracts.merge(
    acs,
    on="GEOID",
    how="left"
)

tracts_acs.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME_x,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,NAME_y,B01003_001E,B19013_001E,state,county,tract
0,48,201,323701,48201323701,1400000US48201323701,3237.01,Census Tract 3237.01,G5020,S,2311208,5694,+29.6551252,-095.1769249,"POLYGON ((-95.18667 29.66507, -95.18633 29.665...",Census Tract 3237.01; Harris County; Texas,4504,54020,48,201,323701
1,48,201,241103,48201241103,1400000US48201241103,2411.03,Census Tract 2411.03,G5020,S,2782048,35540,+30.0474108,-095.3807767,"POLYGON ((-95.3915 30.05522, -95.39134 30.0553...",Census Tract 2411.03; Harris County; Texas,4606,88482,48,201,241103
2,48,201,450801,48201450801,1400000US48201450801,4508.01,Census Tract 4508.01,G5020,S,903291,17758,+29.7598221,-095.5680149,"POLYGON ((-95.57635 29.76008, -95.57634 29.760...",Census Tract 4508.01; Harris County; Texas,1914,80234,48,201,450801
3,48,201,454302,48201454302,1400000US48201454302,4543.02,Census Tract 4543.02,G5020,S,2750427,1867,+29.7153470,-095.6704755,"POLYGON ((-95.68708 29.71008, -95.68669 29.710...",Census Tract 4543.02; Harris County; Texas,5808,71463,48,201,454302
4,48,201,451902,48201451902,1400000US48201451902,4519.02,Census Tract 4519.02,G5020,S,2057671,0,+29.7225162,-095.5987045,"POLYGON ((-95.60573 29.72978, -95.60278 29.729...",Census Tract 4519.02; Harris County; Texas,2593,130469,48,201,451902


## Additional Spatial Analysis

This step supports the comparison between transit corridors and population density.

In [11]:
tracts_acs["population"] = pd.to_numeric(
    tracts_acs["B01003_001E"],
    errors="coerce"
)

tracts_acs["land_sq_miles"] = tracts_acs["ALAND"] / 2589988.11

tracts_acs["pop_density"] = (
    tracts_acs["population"] / tracts_acs["land_sq_miles"]
)

tracts_acs[
    ["GEOID", "population", "land_sq_miles", "pop_density"]
].head()

,GEOID,population,land_sq_miles,pop_density
0,48201323701,4504,0.892362,5047.276769
1,48201241103,4606,1.074155,4288.022793
2,48201450801,1914,0.348763,5487.973690
3,48201454302,5808,1.061946,5469.205670
4,48201451902,2593,0.794471,3263.806104


## 2. Load Harris County Population Density Data

The population density file combines Census tract geometry with ACS population data. This allows the project to map where population is concentrated across Harris County.

In [12]:
tracts_acs.to_file(
    "../data/processed/harris_tracts_density.geojson",
    driver="GeoJSON"
)

print("saved")

saved


## 3. Load Route Geometry and Investment Metrics

Route geometry provides the map coordinates for selected METRO corridors. Investment metrics provide ridership, scheduled service, productivity, and scoring information.

In [13]:
route_geometry = pd.read_csv(
    "../data/processed/key_route_geometry.csv"
)

route_geometry.head()

,route_id,shape_id,route_short_name,route_long_name,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,2,284178,2,Bellaire,29.704383,-95.403284,1,0.0000
1,2,284178,2,Bellaire,29.703900,-95.403170,2,0.0539
2,2,284178,2,Bellaire,29.703849,-95.403190,3,0.0600
3,2,284178,2,Bellaire,29.703770,-95.403249,4,0.0708
4,2,284178,2,Bellaire,29.703750,-95.403409,5,0.0870


## 3. Load Route Geometry and Investment Metrics

Route geometry provides the map coordinates for selected METRO corridors. Investment metrics provide ridership, scheduled service, productivity, and scoring information.

In [14]:
investment = pd.read_csv(
    "../data/processed/route_investment_metrics.csv"
)

investment.head()

,route_long_name,route_type,route_text_color,route_color,agency_id,route_id,route_url,route_desc,route_short_name,avg_weekday_boardings,scheduled_trips,riders_per_trip
0,Richmond,3,FFFFFF,4080,HOU,25,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,25,7489,942.0,7.950106
1,Gessner,3,FFFFFF,4080,HOU,46,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,46,7670,854.0,8.981265
2,Westheimer,3,FFFFFF,4080,HOU,82,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,82,14054,1487.0,9.451244
3,Bellaire,3,FFFFFF,4080,HOU,2,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,2,7590,1013.0,7.492596
4,Beechnut,3,FFFFFF,4080,HOU,4,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,4,8743,896.0,9.757812


## Summary

This notebook adds geographic context to the investment analysis.

The selected high-ridership routes generally overlap with some of the denser parts of Harris County, which strengthens the case for evaluating these corridors as potential targets for future service improvements or capital investment.

This geographic evidence is used in the final recommendations notebook to connect ridership performance with population demand.
